# Model Summary Comparison

Build and compare the model summaries for MobileNetV3Small, CustomCNN, ResNet50, SqueezeNet, and EfficientNetV2B0 without training.

In [8]:
import csv
from contextlib import redirect_stdout
from io import StringIO
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetV2B0, MobileNetV3Small, ResNet50

print('TensorFlow:', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))

TensorFlow: 2.10.0
GPU available: True


## Settings

`USE_IMAGENET_WEIGHTS` is disabled by default to avoid downloading weights. The summaries still show the same architecture and parameter counts.

In [9]:
NUM_CLASSES = 3
CLASS_NAMES = ['Amine', 'Rifki', 'Jakub']
USE_IMAGENET_WEIGHTS = False
APPLICATION_WEIGHTS = 'imagenet' if USE_IMAGENET_WEIGHTS else None

MODEL_CONFIGS = {
    'MobileNetV3Small': {
        'image_size': (160, 160),
        'alpha': 0.75,
        'dropout': 0.2,
        'dense_units': 128,
        'l2': 0.0,
        'learning_rate': 1e-3,
    },
    'CustomCNN': {
        'image_size': (160, 160),
        'base_filters': 32,
        'dense_units': 128,
        'dropout': 0.2,
        'l2': 0.0,
        'learning_rate': 1e-3,
    },
    'ResNet50': {
        'image_size': (224, 224),
        'dropout1': 0.3,
        'dense_units': 128,
        'dropout2': 0.2,
        'learning_rate': 1e-3,
    },
    'SqueezeNet': {
        'image_size': (128, 128),
        'dropout': 0.4,
        'learning_rate': 1e-3,
    },
    'EfficientNetV2B0': {
        'image_size': (224, 224),
        'dropout1': 0.25,
        'dense_units': 128,
        'dropout2': 0.2,
        'learning_rate': 1e-3,
    },
}

MODEL_CONFIGS

{'MobileNetV3Small': {'image_size': (160, 160),
  'alpha': 0.75,
  'dropout': 0.2,
  'dense_units': 128,
  'l2': 0.0,
  'learning_rate': 0.001},
 'CustomCNN': {'image_size': (160, 160),
  'base_filters': 32,
  'dense_units': 128,
  'dropout': 0.2,
  'l2': 0.0,
  'learning_rate': 0.001},
 'ResNet50': {'image_size': (224, 224),
  'dropout1': 0.3,
  'dense_units': 128,
  'dropout2': 0.2,
  'learning_rate': 0.001},
 'SqueezeNet': {'image_size': (128, 128),
  'dropout': 0.4,
  'learning_rate': 0.001},
 'EfficientNetV2B0': {'image_size': (224, 224),
  'dropout1': 0.25,
  'dense_units': 128,
  'dropout2': 0.2,
  'learning_rate': 0.001}}

## Model Builders

In [10]:
def compile_model(model: keras.Model, learning_rate: float) -> keras.Model:
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model


def build_mobilenetv3small_model(config: dict) -> keras.Model:
    image_size = tuple(config['image_size'])
    regularizer = keras.regularizers.l2(config['l2']) if config['l2'] else None
    base_model = MobileNetV3Small(
        include_top=False,
        weights=APPLICATION_WEIGHTS,
        input_shape=image_size + (3,),
        alpha=float(config['alpha']),
    )
    base_model.trainable = False

    inputs = keras.Input(shape=image_size + (3,))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(config['dropout'])(x)
    x = layers.Dense(config['dense_units'], activation='relu', kernel_regularizer=regularizer)(x)
    x = layers.Dropout(config['dropout'])(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', kernel_regularizer=regularizer)(x)
    return compile_model(keras.Model(inputs, outputs, name='MobileNetV3Small_3Class'), config['learning_rate'])


def conv_block(x, filters: int, block_name: str, l2_strength: float):
    regularizer = keras.regularizers.l2(l2_strength) if l2_strength else None
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False, kernel_regularizer=regularizer, name=f'{block_name}_conv1')(x)
    x = layers.BatchNormalization(name=f'{block_name}_bn1')(x)
    x = layers.ReLU(name=f'{block_name}_relu1')(x)
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False, kernel_regularizer=regularizer, name=f'{block_name}_conv2')(x)
    x = layers.BatchNormalization(name=f'{block_name}_bn2')(x)
    x = layers.ReLU(name=f'{block_name}_relu2')(x)
    x = layers.MaxPooling2D(pool_size=2, name=f'{block_name}_pool')(x)
    return x


def build_custom_cnn_model(config: dict) -> keras.Model:
    image_size = tuple(config['image_size'])
    base_filters = int(config['base_filters'])
    l2_strength = float(config['l2'])
    regularizer = keras.regularizers.l2(l2_strength) if l2_strength else None

    inputs = keras.Input(shape=image_size + (3,))
    x = conv_block(inputs, base_filters, 'block1', l2_strength)
    x = conv_block(x, base_filters * 2, 'block2', l2_strength)
    x = conv_block(x, base_filters * 4, 'block3', l2_strength)
    x = layers.SeparableConv2D(base_filters * 6, 3, padding='same', use_bias=False, depthwise_regularizer=regularizer, pointwise_regularizer=regularizer, name='sep_conv1')(x)
    x = layers.BatchNormalization(name='sep_bn1')(x)
    x = layers.ReLU(name='sep_relu1')(x)
    x = layers.SeparableConv2D(base_filters * 8, 3, padding='same', use_bias=False, depthwise_regularizer=regularizer, pointwise_regularizer=regularizer, name='sep_conv2')(x)
    x = layers.BatchNormalization(name='sep_bn2')(x)
    x = layers.ReLU(name='sep_relu2')(x)
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(config['dropout'], name='dropout1')(x)
    x = layers.Dense(config['dense_units'], activation='relu', kernel_regularizer=regularizer, name='dense1')(x)
    x = layers.Dropout(config['dropout'], name='dropout2')(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', kernel_regularizer=regularizer, name='classifier')(x)
    return compile_model(keras.Model(inputs, outputs, name='CustomCNN3Class'), config['learning_rate'])


def build_resnet50_model(config: dict) -> keras.Model:
    image_size = tuple(config['image_size'])
    base_model = ResNet50(include_top=False, weights=APPLICATION_WEIGHTS, input_shape=image_size + (3,))
    base_model.trainable = False

    inputs = keras.Input(shape=image_size + (3,))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(config['dropout1'])(x)
    x = layers.Dense(config['dense_units'], activation='relu')(x)
    x = layers.Dropout(config['dropout2'])(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    return compile_model(keras.Model(inputs, outputs, name='ResNet50_3Class'), config['learning_rate'])


def fire_module(x, squeeze_filters: int, expand_filters: int, name: str):
    squeezed = layers.Conv2D(squeeze_filters, (1, 1), activation='relu', padding='same', name=f'{name}_squeeze')(x)
    expand_1x1 = layers.Conv2D(expand_filters, (1, 1), activation='relu', padding='same', name=f'{name}_expand1x1')(squeezed)
    expand_3x3 = layers.Conv2D(expand_filters, (3, 3), activation='relu', padding='same', name=f'{name}_expand3x3')(squeezed)
    return layers.Concatenate(name=f'{name}_concat')([expand_1x1, expand_3x3])


def build_squeezenet_model(config: dict) -> keras.Model:
    image_size = tuple(config['image_size'])
    inputs = keras.Input(shape=image_size + (3,))
    x = layers.Conv2D(64, (3, 3), strides=2, padding='same', activation='relu', name='conv1')(inputs)
    x = layers.MaxPooling2D(pool_size=(3, 3), strides=2, padding='same', name='maxpool1')(x)
    x = fire_module(x, 16, 64, 'fire2')
    x = fire_module(x, 16, 64, 'fire3')
    x = layers.MaxPooling2D(pool_size=(3, 3), strides=2, padding='same', name='maxpool3')(x)
    x = fire_module(x, 32, 128, 'fire4')
    x = fire_module(x, 32, 128, 'fire5')
    x = layers.MaxPooling2D(pool_size=(3, 3), strides=2, padding='same', name='maxpool5')(x)
    x = fire_module(x, 48, 192, 'fire6')
    x = fire_module(x, 48, 192, 'fire7')
    x = fire_module(x, 64, 256, 'fire8')
    x = fire_module(x, 64, 256, 'fire9')
    x = layers.Dropout(config['dropout'], name='dropout')(x)
    x = layers.Conv2D(NUM_CLASSES, (1, 1), padding='same', activation='relu', name='classifier_conv')(x)
    x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    outputs = layers.Activation('softmax', name='predictions')(x)
    return compile_model(keras.Model(inputs, outputs, name='SqueezeNetKeras'), config['learning_rate'])


def build_efficientnetv2b0_model(config: dict) -> keras.Model:
    image_size = tuple(config['image_size'])
    data_augmentation = keras.Sequential([layers.Rescaling(1.0 / 255.0)], name='data_rescaling')
    base_model = EfficientNetV2B0(include_top=False, weights=APPLICATION_WEIGHTS, input_shape=image_size + (3,))
    base_model.trainable = False

    inputs = keras.Input(shape=image_size + (3,))
    x = data_augmentation(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(config['dropout1'])(x)
    x = layers.Dense(config['dense_units'], activation='relu')(x)
    x = layers.Dropout(config['dropout2'])(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    return compile_model(keras.Model(inputs, outputs, name='EfficientNetV2B0_3Class'), config['learning_rate'])

In [11]:
MODEL_BUILDERS = {
    'MobileNetV3Small': build_mobilenetv3small_model,
    'CustomCNN': build_custom_cnn_model,
    'ResNet50': build_resnet50_model,
    'SqueezeNet': build_squeezenet_model,
    'EfficientNetV2B0': build_efficientnetv2b0_model,
}

## Summary Table

In [12]:
def trainable_parameter_count(model: keras.Model) -> int:
    return int(sum(keras.backend.count_params(w) for w in model.trainable_weights))


def non_trainable_parameter_count(model: keras.Model) -> int:
    return int(sum(keras.backend.count_params(w) for w in model.non_trainable_weights))


models = {}
rows = []
for model_name, builder in MODEL_BUILDERS.items():
    keras.backend.clear_session()
    model = builder(MODEL_CONFIGS[model_name])
    models[model_name] = model
    rows.append({
        'model': model_name,
        'keras_name': model.name,
        'input_shape': model.input_shape,
        'output_shape': model.output_shape,
        'layers': len(model.layers),
        'total_params': model.count_params(),
        'trainable_params': trainable_parameter_count(model),
        'non_trainable_params': non_trainable_parameter_count(model),
    })

for row in rows:
    print(
        f"{row['model']:<18} "
        f"input={row['input_shape']} "
        f"output={row['output_shape']} "
        f"layers={row['layers']:<4} "
        f"total={row['total_params']:,} "
        f"trainable={row['trainable_params']:,} "
        f"non_trainable={row['non_trainable_params']:,}"
    )

rows

MobileNetV3Small   input=(None, 160, 160, 3) output=(None, 3) layers=7    total=638,971 trainable=55,811 non_trainable=583,160
CustomCNN          input=(None, 160, 160, 3) output=(None, 3) layers=33   total=400,035 trainable=398,243 non_trainable=1,792
ResNet50           input=(None, 224, 224, 3) output=(None, 3) layers=7    total=23,850,371 trainable=262,659 non_trainable=23,587,712
SqueezeNet         input=(None, 128, 128, 3) output=(None, 3) layers=41   total=724,035 trainable=724,035 non_trainable=0
EfficientNetV2B0   input=(None, 224, 224, 3) output=(None, 3) layers=8    total=6,083,667 trainable=164,355 non_trainable=5,919,312


[{'model': 'MobileNetV3Small',
  'keras_name': 'MobileNetV3Small_3Class',
  'input_shape': (None, 160, 160, 3),
  'output_shape': (None, 3),
  'layers': 7,
  'total_params': 638971,
  'trainable_params': 55811,
  'non_trainable_params': 583160},
 {'model': 'CustomCNN',
  'keras_name': 'CustomCNN3Class',
  'input_shape': (None, 160, 160, 3),
  'output_shape': (None, 3),
  'layers': 33,
  'total_params': 400035,
  'trainable_params': 398243,
  'non_trainable_params': 1792},
 {'model': 'ResNet50',
  'keras_name': 'ResNet50_3Class',
  'input_shape': (None, 224, 224, 3),
  'output_shape': (None, 3),
  'layers': 7,
  'total_params': 23850371,
  'trainable_params': 262659,
  'non_trainable_params': 23587712},
 {'model': 'SqueezeNet',
  'keras_name': 'SqueezeNetKeras',
  'input_shape': (None, 128, 128, 3),
  'output_shape': (None, 3),
  'layers': 41,
  'total_params': 724035,
  'trainable_params': 724035,
  'non_trainable_params': 0},
 {'model': 'EfficientNetV2B0',
  'keras_name': 'EfficientNe

## Full Model Summaries

In [13]:
summary_texts = {}
for model_name, model in models.items():
    print('\n' + '=' * 100)
    print(model_name)
    print('=' * 100)
    buffer = StringIO()
    model.summary(print_fn=lambda line: buffer.write(line + '\n'))
    summary_text = buffer.getvalue()
    summary_texts[model_name] = summary_text
    print(summary_text)


MobileNetV3Small
Model: "MobileNetV3Small_3Class"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 160, 160, 3)]     0         
                                                                 
 MobilenetV3small (Functiona  (None, 5, 5, 432)        583160    
 l)                                                              
                                                                 
 global_average_pooling2d (G  (None, 432)              0         
 lobalAveragePooling2D)                                          
                                                                 
 dropout (Dropout)           (None, 432)               0         
                                                                 
 dense (Dense)               (None, 128)               55424     
                                                                 
 dropout_1 (Dropout)     

## Save Summaries

In [14]:
OUTPUT_DIR = Path('model_summaries')
OUTPUT_DIR.mkdir(exist_ok=True)

with (OUTPUT_DIR / 'model_summary_comparison.csv').open('w', newline='', encoding='utf-8') as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

for model_name, summary_text in summary_texts.items():
    (OUTPUT_DIR / f'{model_name}_summary.txt').write_text(summary_text, encoding='utf-8')

print(f'Saved summaries to: {OUTPUT_DIR.resolve()}')

Saved summaries to: C:\Users\Rifki\Personal\CollegeWorkspace\HardwareSoftwareCo\Project\H-S-Codesign\python\model_summaries
